In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as integrate
from scipy.stats import gaussian_kde
plt.rcParams['text.usetex'] = True

In [ ]:
# =========================
# Parameters
# =========================
Nn = 100
m = 1.0
beta = 1
lam = 1/np.sqrt(Nn)
U0 = 0.5
Ul = U0 * lam
g = 5
wDrive = 0.5
V0 = 10

dt = 5e-3
tmax = 2000
nSteps = int(np.ceil(tmax / dt))
Psi0 = 1
times = np.linspace(0,tmax,nSteps+1)

In [ ]:
# =========================
# Core functions
# =========================
def kRate(x, eta, eta_prime):
    if eta == eta_prime:
        return 0.0
    # Reactivities
    if eta == "a" and eta_prime == "b":
        psi = Psi0 * (1 + lam * g * np.exp(-x**2))
    elif eta == "b" and eta_prime == "a":
        psi = Psi0 * (1 + lam * g * np.exp(-x**2))
    else:
        psi = Psi0

    return psi*np.exp(sX(x, eta, eta_prime)/2.0)
    
def sX(x, eta, eta_prime):
    Ueta = UInt(x) if eta == "a" else 0.0
    Ueta_p = UInt(x) if eta_prime == "a" else 0.0
    return beta * (Ueta - Ueta_p+ W_term(eta, eta_prime))

def cyclic_next(s):
    return {"a": "b", "b": "c", "c": "a"}[s]

def cyclic_prev(s):
    return cyclic_next(cyclic_next(s))

def W_term(eta, eta_prime):
    if eta_prime == cyclic_next(eta):
        return wDrive
    elif eta_prime == cyclic_prev(eta):
        return -wDrive
    else:
        return 0.0

def potV(x):
    return V0*x**2/2

def UInt(x):
    return Ul*x**2/2

def dU(x):
    return Ul*x

def integratedUPsi(x):
    return Ul*g/2*(1-np.exp(-x**2))

def Ueff(x):
    return potV(x)+lam*Nn/3*UInt(x)-lam**2*Nn*(np.cosh(beta*wDrive)+1)/(3*(2*np.cosh(beta*wDrive)+1))*beta*UInt(x)**2/2-lam**2*Nn*(3*np.sinh(beta*wDrive)+np.cosh(beta*wDrive)-1)/(9*(2*np.cosh(beta*wDrive)+1))*integratedUPsi(x)

def Psi(x):
    return Psi0*g*lam*np.exp(-x**2)
    
def dPsi(x):
    return -2*Psi0*g*lam*x*np.exp(-x**2)

In [ ]:
# =========================
# Vectorized Simulation
# =========================

def simulate(init_state, init_x, init_v):
    x = init_x
    v = init_v
    eta = np.array(init_state, dtype=object)
    t = 0.0

    times = np.zeros(nSteps + 1)
    xs = np.zeros(nSteps + 1)
    vs = np.zeros(nSteps + 1)
    etas = np.empty((nSteps + 1, Nn), dtype=object)

    xs[0] = x
    vs[0] = v
    etas[0] = eta.copy()

    for i in range(nSteps):

        # ===== Velocity Verlet =====
        # Force at current (x, η)
        nA = np.sum(eta == "a")
        acc = (-V0*x - nA*Ul*x)/m
        
        # Update everything forward
        v = v + acc * dt
        x = (x + v * dt)   

        # ===== Compute rates at current x =====
        rA = np.array([kRate(x, "a", "b"), kRate(x, "a", "c")])
        rB = np.array([kRate(x, "b", "a"), kRate(x, "b", "c")])
        rC = np.array([kRate(x, "c", "a"), kRate(x, "c", "b")])

        totalRates = {
            "a": np.sum(rA),
            "b": np.sum(rB),
            "c": np.sum(rC)
        }

        # ===== Vectorized jump decision =====
        total_vec = np.vectorize(totalRates.get)(eta)
        jump_probs = 1 - np.exp(-total_vec * dt)
        jump_mask = np.random.rand(Nn) < jump_probs

        rands = np.random.rand(Nn)
        u = rands * total_vec

        # Masks per state
        maskA = (eta == "a") & jump_mask
        maskB = (eta == "b") & jump_mask
        maskC = (eta == "c") & jump_mask

        # Allocate new states
        new_eta = eta.copy()

        # State A transitions
        new_eta[maskA & (u < rA[0])] = "b"
        new_eta[maskA & (u >= rA[0])] = "c"

        # State B transitions
        new_eta[maskB & (u < rB[0])] = "a"
        new_eta[maskB & (u >= rB[0])] = "c"

        # State C transitions
        new_eta[maskC & (u < rC[0])] = "a"
        new_eta[maskC & (u >= rC[0])] = "b"

        eta = new_eta

        # ===== Store =====
        t += dt
        times[i + 1] = t
        xs[i + 1] = x
        vs[i + 1] = v
        etas[i + 1] = eta.copy()

    return {
        "times": times,
        "x": xs,
        "v": vs,
        "eta": etas
    }

## Plots of rates and induced functions ##

In [ ]:
# =========================
# Rates plot
# =========================
xL=np.linspace(-5,5,1000)
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(xL, kRate(xL,"a","b"), label = "Total Rate")
plt.plot(xL,Psi0 * (1 + lam * g * np.exp(-xL**2)),  label = "Reactivity")
ax.set_ylabel(r"Rates", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
any(val > 0 for val in 1+Psi(xL))

In [ ]:
# =========================
# Reactivity plot seperately
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
#plt.hist(xredstation, bins = "auto", density = True, label = "Reduced dynamics")
plt.plot(xL, 1+Psi(xL))
plt.plot(xL,Psi0 * (1 + lam * g * np.exp(-xL**2)))
ax.set_ylabel(r"$\psi(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
#ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# \nu(x) plot
# =========================
A=2*beta*np.cosh(beta*wDrive/2)*(np.cosh(beta*wDrive)+2)
B = np.exp(-beta*wDrive/2)*(np.exp(2*beta*wDrive)+np.exp(beta*wDrive)-2)
xL=np.linspace(-5,5,1000)
nu=(A*(dU(xL))**2+B*dU(xL)*dPsi(xL))*1/(9*(2*np.cosh(beta*wDrive)+1)**2)*Nn
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(xL, nu, label = "Friction coefficient")
ax.set_ylabel(r"$\nu(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
not min(nu) < 0 < max(nu)

In [ ]:
any(val < 0 for val in nu)

In [ ]:
any(val > 0 for val in nu)

In [ ]:
# =========================
# Potentials plot
# =========================
xL=np.linspace(-5,5,1000)
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(xL, Ueff(xL), label = "Effective potential")
plt.plot(xL, potV(xL), label = "Uncoupled potential")
ax.set_ylabel(r"Potentials", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

# Equilibrium results #

In [ ]:
# =========================
# Equilibrium functions
# =========================
def rho1(x):
    return np.exp(-beta*potV(x))*intexpU(x)**Nn
def marginalv(v):
    return np.exp(-beta*m*v**2/(2))/np.sqrt(2*np.pi/(m*beta))
def rho_x(x):
   return rho1(x)/Z1
def marginalx(x):
    return 1/Zx*np.exp(-beta*potV(x))
def intexpU(x):
    return 1+1+np.exp(-beta*UInt(x)) #integrate.quad(lambda y: np.exp(-beta*UInt(x)),0,L)[0]

Zx=integrate.quad(lambda x: np.exp(-beta*potV(x)), -10, 10)[0]
Z1=integrate.quad(lambda x: rho1(x), -10, 10)[0]

In [ ]:
# =========================
# Equilibrium expectations
# =========================
xline = np.linspace(-10,10,1000)
vline = np.linspace(-10,10,1000)
rho_x_vals = np.array([rho_x(x) for x in xline])
Vx=potV(xline)
minindex=np.where(Vx == Vx.min())[0][0]
xmin=xline[minindex]
avx=integrate.quad(lambda x: x*rho_x(x), -10, 10)[0]
avx2= integrate.quad(lambda x: x**2*rho_x(x), -10, 10)[0]
varx=avx2-avx**2

In [ ]:
# =========================
# Equilibrium position distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(xline, rho_x_vals, label = 'coupled distribution')
plt.plot(xline, marginalx(xline), label = 'uncoupled distribution')
#plt.plot(xline, marginalx(xline))
plt.axvline(x = xmin, color = 'b', label = 'Location minimum of potential', linestyle = '--')
ax.set_ylabel(r"$\rho^\mathrm{stat}(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Equilibrium velocity distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(vline, marginalv(vline))
ax.set_ylabel(r"$\rho^\mathrm{stat}(v)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$v$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
#ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

# Dynamics for 1 run #

In [ ]:
eta0 = np.random.choice(["a", "b", "c"], size=Nn)
x0 = 2
v0 = 1
onerun = simulate(eta0, x0, v0)

In [ ]:
# =========================
# x(t) for 1 run
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(onerun["times"], onerun["x"])
ax.set_ylabel(r"$x(t)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# v(t) for 1 run
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(onerun["times"], onerun["v"])
ax.set_ylabel(r"$v(t)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

# Dynamics over many runs #

In [ ]:
x0 = 2
v0 = 1
xc=np.sqrt(x0**2+m*v0**2/V0)
vc = np.sqrt(v0**2+V0*x0**2/m)
trials = 20
x_all = np.zeros((trials, nSteps + 1))
v_all = np.zeros((trials, nSteps + 1))

for i in range(trials):
    # Random initial states
    eta0 = np.random.choice(["a", "b", "c"], size=Nn)
    
    # Run simulation
    res = simulate(eta0, x0, v0)
    x_all[i, :] = res["x"]
    v_all[i, :] = res["v"]
    
    print(f"Trial {i+1} done")

x_mean = np.mean(x_all, axis=0)
v_mean = np.mean(v_all, axis=0)

# Standard deviation (for envelope)
x_std = np.std(x_all, axis=0)
v_std = np.std(v_all, axis=0)

In [ ]:
# =========================
# Average x(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(times, x_mean,label='Simulation', color = 'blue')
plt.plot(times,times*0+avx,label='Equilibrium expectation',linestyle='--', color = 'lime')
ax.set_ylabel(r"$\langle x(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.legend(fontsize = 12, loc = 'upper right')
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Standard deviation x(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, x_std, label='Simulation', color = 'blue')
ax.set_xlabel(r"$t$", fontsize = 15)
ax.set_ylabel(r"$\langle s_{x}(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=45)
ax.plot(times, times*0+np.sqrt(varx), label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Average v(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(times, v_mean,label='Simulation', color = 'blue')
ax.set_ylabel(r"$\langle v(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.plot(times, times*0, label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Standard deviation v(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, v_std, label='Simulation', color = 'blue')
ax.set_xlabel(r"$t$", fontsize = 15)
ax.set_ylabel(r"$\langle s_{v}(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=45)
ax.plot(times, times*0+np.sqrt(1/(m*beta)), label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Keep stationary values only
# =========================
vstation = v_all[:,-200000:].reshape(-1)
xstation = x_all[:,-200000:].reshape(-1)
vv=np.linspace(min(vstation), max(vstation), 10000)
xx = np.linspace(min(xstation), max(xstation), 10000)
rho_x_valsxx = np.array([rho_x(x) for x in xx])

In [ ]:
# =========================
# Stationary position distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.hist(xstation, density = True, bins = "auto", label = "Simulation")
plt.plot(xx, rho_x_valsxx, label = "Equilibrium Distribution",color = "black", linestyle = "--")
ax.set_ylabel(r"$\rho^\mathrm{stat}(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Kernel density estimation
# =========================
kdex = gaussian_kde(xstation)
x_range = np.linspace(min(xstation), max(xstation), 500)
normx, _ = integrate.quad(kdex, min(xstation), max(xstation))
print(normx)

In [ ]:
# =========================
# Estimated stationary position distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_range/xc, xc*kdex(x_range), label = "Simulation")
ax.set_ylabel(r"$x_c \ \rho^{\mathrm{stat}}(x)$", fontsize=25, rotation=0, labelpad=70)
ax.set_xlabel(r"$x/x_c$", fontsize=30)
plt.tight_layout()
ax.spines[['right', 'top']].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=15)
for axis in 'left', 'bottom':
    ax.spines[axis].set_linewidth(1.5)
plt.show()

In [ ]:
# =========================
# Stationary velocity distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.hist(vstation, density = True, bins = "auto", label = "Simulation")
plt.plot(vv, marginalv(vv), label = "Equilibrium Distribution",color = "black", linestyle = "--")
ax.set_ylabel(r"$\rho^\mathrm{stat}(v)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$v$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Kernel density estimation
# =========================
kdev = gaussian_kde(vstation)
v_range = np.linspace(min(vstation), max(vstation), 500)
normv, _ = integrate.quad(kdev, min(vstation), max(vstation))
print(normv)

In [ ]:
# =========================
# Estimated stationary velocity distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(v_range/vc, kdev(v_range)*vc, label = "Simulation")
ax.set_ylabel(r"$v_{c} \ \rho^{\mathrm{stat}}(v)$", fontsize=25, rotation=0, labelpad=70)
ax.set_xlabel(r"$v/v_{c}$", fontsize=30)
plt.tight_layout()
ax.spines[['right', 'top']].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=15)
for axis in 'left', 'bottom':
    ax.spines[axis].set_linewidth(1.5)
#plt.savefig('final line negative friction velocity distribution 100.png')
plt.show()

## Negative friction regime with driving ##

In [ ]:
# =========================
# Parameters
# =========================
Nn = 100
m = 1.0
beta = 1
lam = 1/np.sqrt(Nn)
U0 = 0.5
Ul = U0 * lam
g = 5
wDrive = 0.5
V0 = 10

dt = 5e-3
tmax = 2000
nSteps = int(np.ceil(tmax / dt))
Psi0 = 1
times = np.linspace(0,tmax,nSteps)
x0 = 2
v0 = 1
xc=np.sqrt(x0**2+m*v0**2/V0)
vc = np.sqrt(v0**2+V0*x0**2/m)

In [ ]:
# =========================
# \nu(x) plot
# =========================
A=2*beta*np.cosh(beta*wDrive/2)*(np.cosh(beta*wDrive)+2)
B = np.exp(-beta*wDrive/2)*(np.exp(2*beta*wDrive)+np.exp(beta*wDrive)-2)
xL=np.linspace(-5,5,1000)
nu=(A*(dU(xL))**2+B*dU(xL)*dPsi(xL))*1/(9*(2*np.cosh(beta*wDrive)+1)**2)*Nn
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(xL, nu, label = "Friction coefficient")
ax.set_ylabel(r"$\nu(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
not min(nu) < 0 < max(nu)

In [ ]:
any(val < 0 for val in nu)

In [ ]:
any(val > 0 for val in nu)

In [ ]:
# =========================
# Data
# =========================
x_allneqlineneg100 = np.load("x_allneqlineneg100.npy")
v_allneqlineneg100 = np.load("v_allallneqlineneg.npy")
timesneqlineneg100 = np.load("timeslineneg100.npy")


xneqlineneg100_mean = np.mean(x_allneqlineneg100, axis=0)
vneqlineneg100_mean = np.mean(v_allneqlineneg100, axis=0)

xneqlineneg100_std = np.std(x_allneqlineneg100, axis=0)
vneqlineneg100_std = np.std(v_allneqlineneg100, axis=0)

In [ ]:
# =========================
# Average x(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(timesneqlineneg100, xneqlineneg100_mean,label='Simulation', color = 'blue')
plt.plot(timesneqlineneg100,timesneqlineneg100*0+avx,label='Equilibrium expectation',linestyle='--', color = 'lime')
ax.set_ylabel(r"$\langle x(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.legend(fontsize = 12, loc = 'upper right')
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Standard deviation x(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timesneqlineneg100, xneqlineneg100_std, label='Simulation', color = 'blue')
ax.set_xlabel(r"$t$", fontsize = 15)
ax.set_ylabel(r"$\langle s_{x}(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=45)
ax.plot(times, times*0+np.sqrt(varx), label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Average v(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.plot(timesneqlineneg100, vneqlineneg100_mean,label='Simulation', color = 'blue')
ax.set_ylabel(r"$\langle v(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$t$", fontsize = 15)
ax.plot(times, times*0, label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Standard deviation v(t) over many runs
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timesneqlineneg100, vneqlineneg100_std , label='Simulation', color = 'blue')
ax.set_xlabel(r"$t$", fontsize = 15)
ax.set_ylabel(r"$\langle s_{v}(t) \rangle_\sigma$", fontsize = 15, rotation = 0,labelpad=45)
ax.plot(times, times*0+np.sqrt(1/(m*beta)), label='Equilibrium expectation', linestyle='--', color = 'lime')
ax.legend(fontsize = 12)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Keep stationary values only
# =========================
vstationlineneg100 = v_allneqlineneg100[:,-50000:].reshape(-1)
xstationlineneg100 = x_allneqlineneg100[:,-50000:].reshape(-1)
vvlineneg100=np.linspace(min(vstationlineneg100), max(vstationlineneg100), 10000)
xxlineneg100 = np.linspace(min(xstationlineneg100), max(xstationlineneg100), 10000)
rho_x_valsxxlineneg100 = np.array([rho_x(x) for x in xxlineneg100])

In [ ]:
# =========================
# Stationary position distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.hist(xstationlineneg100, density = True, bins = "auto", label = "Simulation")
ax.set_ylabel(r"$\rho^\mathrm{stat}(x)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$x$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Kernel density estimation
# =========================
kdexlineneg100 = gaussian_kde(xstationlineneg100)
x_rangelineneg100 = np.linspace(min(xstationlineneg100), max(xstationlineneg100), 500)
normxlineneg100, _ = quad(kdexlineneg100, min(xstationlineneg100), max(xstationlineneg100))
print(normxlineneg100)

In [ ]:
# =========================
# Estimated stationary position distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_rangelineneg100/xc, kdexlineneg100(x_rangelineneg100)*xc, label = "Distribution")
ax.plot(x_rangelineneg100/xc, Ueff(x_rangelineneg100)/(3*V0/2*xc**2), label = r"$V_{\mathrm{eff}}(x)/(3 V_0)$", linestyle = ':', color = 'black')
ax.set_ylabel(r"$x_c \ \rho^{\mathrm{stat}}(x)$", fontsize=25, rotation=0, labelpad=70)
ax.set_xlabel(r"$x/x_c$", fontsize=30)
plt.tight_layout()
ax.spines[['right', 'top']].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=15)
for axis in 'left', 'bottom':
    ax.spines[axis].set_linewidth(1.5)
ax.legend(fontsize = 15)
ax.spines['bottom'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))

ax.spines['top'].set_visible(False)
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.tick_params(axis='x', pad=8)
#plt.savefig('final line negative friction position distribution 100 marco.png')
plt.show()

In [ ]:
# =========================
# Stationary velocity distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
plt.hist(vstationlineneg100, density = True, bins = "auto", label = "Simulation")
ax.set_ylabel(r"$\rho^\mathrm{stat}(v)$", fontsize = 15, rotation = 0,labelpad=30)
ax.set_xlabel(r"$v$", fontsize = 15)
ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
ax.legend(fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Kernel density estimation
# =========================
kdevlineneg100 = gaussian_kde(vstationlineneg100)
v_rangelineneg100 = np.linspace(min(vstationlineneg100), max(vstationlineneg100), 500)
normvlineneg100, _ = quad(kdevlineneg100, min(vstationlineneg100), max(vstationlineneg100))
print(normvlineneg100)

In [ ]:
# =========================
# Estimated stationary velocity distribution
# =========================
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(v_rangelineneg100/vc, kdevlineneg100(v_rangelineneg100)*vc, label = "Simulation")
ax.set_ylabel(r"$v_{c} \ \rho^{\mathrm{stat}}(v)$", fontsize=25, rotation=0, labelpad=70)
ax.set_xlabel(r"$v/v_{c}$", fontsize=30)
plt.tight_layout()
ax.spines[['right', 'top']].set_visible(False)
ax.tick_params(axis='both', which='major', labelsize=15)
for axis in 'left', 'bottom':
    ax.spines[axis].set_linewidth(1.5)
ax.spines['bottom'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))

ax.spines['top'].set_visible(False)
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.tick_params(axis='x', pad=8)
plt.axvline(x=0, color='black', label='axvline - full height', linestyle = 'dashed')
#plt.savefig('final line negative friction velocity distribution 100 marco.png')
plt.show()

In [ ]:
#np.save("x_allneqlineneg100.npy", x_all)
#np.save("v_allallneqlineneg.npy", v_all)
#np.save("timeslineneg100.npy", times)